# Example 2: Network Quantization and Cox Regression

This notebook demonstrates:
- `quantize_matrix_fast()` — converts continuous expression into discrete activity states
- `cox_regulon()` — tests whether a regulon's activity predicts patient survival

## Background

**Network quantization** converts the continuous eigengene values into a discrete
three-state representation: overactive (+1), neutral (0), or underactive (-1).
This is done per patient by ranking all genes and testing whether a regulon's genes
cluster significantly in the upper or lower third of the expression distribution. 
This is done for three reasons:

1) To ensure data from different cohorts (e.g: TCGA and Gravendeel) are comparable, as they may have been processed differently (e.g: differences in z-scoring).
   
2) To reduce noise: raw expression data for 2 patients can be  different despite similar biology, due to sample processing, batch effects, etc. For example we use a combination of RNA-seq data and microarrays. A gene that appears to be overexpressed in RNA-seq might look to be so simply because RNA-seq gives higher raw counts than microarray

3) Reduces dimensionality: collapses thousands of genes per patient into discrete data which makes downstream processing (e.g: Cox model, risk prediction, etc. easier to handle)

A **binomial test** is used: if a regulon has N genes and k of them fall in the
upper third, we test whether k is significantly more than expected by chance (N/3).
p ≤ 0.05 → regulon is overactive (+1) or underactive (-1) in that patient.

**Cox regression** then tests whether the eigengene of a regulon predicts survival.
Regulons with p ≤ 0.05 are flagged as disease-relevant. High HR (>= 1) indicates high-risk (high eigengene).

In [3]:
import numpy as np
import pandas as pd
import sys
sys.path.append(r"D:\School\IITD\General\GBM\gbm_model.ipynb")

from utils import quantize_matrix_fast, cox_regulon, get_eigengene

## 1. Create synthetic data

In [4]:
np.random.seed(42)
n_genes    = 100
n_patients = 50

gene_names    = [f'GENE_{i}' for i in range(n_genes)]
patient_names = [f'PATIENT_{i}' for i in range(n_patients)]

expr = pd.DataFrame(
    np.random.randn(n_genes, n_patients),
    index=gene_names,
    columns=patient_names
)

# 3 regulons: each is a list of gene names
regulons = [
    gene_names[0:8],    # regulon 0: 8 genes
    gene_names[8:15],   # regulon 1: 7 genes
    gene_names[15:20],  # regulon 2: 5 genes
]

# Inject a strong signal into regulon 0 for first 25 patients
for gene in regulons[0]:
    expr.loc[gene, patient_names[:25]] += 3.0   # overexpressed in half the patients

print(f"Expression matrix: {expr.shape}")
print(f"Number of regulons: {len(regulons)}")

Expression matrix: (100, 50)
Number of regulons: 3


## 2. Quantize the matrix

`quantize_matrix_fast(expr_df, regulon_indices, validated)` takes:
- `expr_df`: full expression matrix (genes × patients)
- `regulon_indices`: list of integers indexing into `validated`
- `validated`: list of lists (each inner list = gene names for that regulon)

For each patient, it ranks all genes once, then tests each regulon
using a binomial test on the upper/lower thirds of the ranking.

In [5]:
regulon_indices = list(range(len(regulons)))   # [0, 1, 2]

quantized = quantize_matrix_fast(expr, regulon_indices, regulons)

print(f"Quantized matrix shape: {quantized.shape}")
print(f"  Rows = regulons, Columns = patients")
print(f"\nValue counts:")
print(quantized.values.flatten())
import pandas as pd
print(pd.Series(quantized.values.flatten()).value_counts())
print(f"\nQuantized matrix preview:")
print(quantized.iloc[:, :8])

Patient 0/50...
Quantized matrix shape: (3, 50)
  Rows = regulons, Columns = patients

Value counts:
[ 1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  1  0  0  0  0  0  0  0
  0  0 -1  0  0 -1  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0 -1  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0
  0  0  0  0  0  0]
 0    117
 1     28
-1      5
Name: count, dtype: int64

Quantized matrix preview:
   PATIENT_0  PATIENT_1  PATIENT_2  PATIENT_3  PATIENT_4  PATIENT_5  \
0          1          1          1          1          1          1   
1          0          0          0          0          0          0   
2          0         -1          0          1          0          0   

   PATIENT_6  PATIENT_7  
0          1          1  
1          0         

### Interpreting the output

Each cell is one regulon × one patient:
- `+1` → regulon genes significantly overexpressed in this patient
- `-1` → regulon genes significantly underexpressed
- ` 0` → no significant deviation from expected

Regulon 0 should show more +1 values in the first 25 patients
since we injected a strong upward signal there.

In [6]:
# Check regulon 0 activity across patient groups
reg0 = quantized.loc[0]
print("Regulon 0 activity in patients 0-24 (signal injected):")
print(reg0[patient_names[:25]].value_counts())
print("\nRegulon 0 activity in patients 25-49 (no signal):")
print(reg0[patient_names[25:]].value_counts())

Regulon 0 activity in patients 0-24 (signal injected):
0
1    25
Name: count, dtype: int64

Regulon 0 activity in patients 25-49 (no signal):
0
0    25
Name: count, dtype: int64


## 3. Cox regression on regulon eigengenes

`cox_regulon(eigen, clinical)` fits a Cox proportional hazards model
asking: does this regulon's eigengene predict how quickly a patient dies?

**Inputs:**
- `eigen`: pd.Series of eigengene scores (one per patient)
- `clinical`: DataFrame with OS_MONTHS and OS_STATUS columns

**Outputs:**
- `hr`: hazard ratio (>1 = high eigengene → worse survival)
- `pval`: p-value (≤ 0.05 = disease-relevant regulon)

In [7]:
# Simulate survival data
# Patients 0-24 (high regulon 0 activity) have shorter survival
np.random.seed(42)
os_months = np.concatenate([
    np.random.exponential(scale=10, size=25),   # short survival — high risk group
    np.random.exponential(scale=30, size=25)    # long survival  — low risk group
])
os_status = np.ones(n_patients, dtype=int)      # all patients deceased for simplicity

clinical = pd.DataFrame({
    'OS_MONTHS': os_months,
    'OS_STATUS': os_status
}, index=patient_names)

print(f"Clinical data shape: {clinical.shape}")
print(clinical.head())

Clinical data shape: (50, 2)
           OS_MONTHS  OS_STATUS
PATIENT_0   4.692681          1
PATIENT_1  30.101214          1
PATIENT_2  13.167457          1
PATIENT_3   9.129426          1
PATIENT_4   1.696249          1


In [8]:
# Compute eigengenes and run Cox for each regulon
print(f"{'Regulon':<10} {'HR':<10} {'p-value':<12} {'Disease-relevant'}")
print("-" * 45)

for idx, regulon in enumerate(regulons):
    eigen = get_eigengene(regulon, expr)
    if eigen is not None:
        hr, pval = cox_regulon(eigen, clinical)
        if hr is not None:
            relevant = '✓' if pval <= 0.05 else ''
            print(f"{idx:<10} {hr:<10.4f} {pval:<12.4f} {relevant}")

# Regulon 0 should be disease-relevant (p ≤ 0.05) since we injected
# a signal that correlates with shorter survival

Regulon    HR         p-value      Disease-relevant
---------------------------------------------
0          1.0897     0.0090       ✓
1          0.7802     0.0468       ✓
2          1.1987     0.1682       


### Interpreting Cox results

- **HR > 1**: higher eigengene activity → worse survival (high-risk regulon)
- **HR < 1**: higher eigengene activity → better survival (protective regulon)
- **p ≤ 0.05**: the association is statistically significant → disease-relevant

In the full GBM pipeline, 181 of 899 regulons pass this threshold
in TCGA, and 148 are confirmed in both TCGA and Gravendeel.

In [9]:
# Edge cases handled by cox_regulon
from utils import cox_regulon

# Too few patients after dropping NaN
tiny_clinical = clinical.iloc[:5]
eigen = get_eigengene(regulons[0], expr)
hr, pval = cox_regulon(eigen, tiny_clinical)
print(f"Too few patients → hr={hr}, pval={pval}")   # None, None

# NaN in survival data — handled by dropna inside function
clinical_with_nan = clinical.copy()
clinical_with_nan.loc['PATIENT_0', 'OS_MONTHS'] = np.nan
hr, pval = cox_regulon(eigen, clinical_with_nan)
print(f"NaN survival data → hr={hr:.4f}, pval={pval:.4f}")  # still works, drops NaN row

Too few patients → hr=None, pval=None
NaN survival data → hr=1.0880, pval=0.0109
